# MCMC H(z) Only (Paper I Dataset Separation)**Author**: Ricardo Alvim**Purpose**: Constrain Evaporating Universe using Cosmic Chronometers H(z) data alone---## Runtime: ~2-3 hours on Colab Pro

In [ ]:
# ============================================================
# INSTALLATION (run this cell first!)
# ============================================================
# Core packages are pre-installed in Colab
# !pip install numpy matplotlib scipy  # Already in Colab
print('Colab environment ready!')

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport emceefrom multiprocessing import Pool, cpu_countimport cornerimport jsonfrom datetime import datetimeimport warningswarnings.filterwarnings('ignore')print("="*70)print("MCMC H(z) ONLY - Evaporating Universe")print("="*70)n_cores = min(cpu_count(), 32)print(f'Available CPU cores: {n_cores}')

In [ ]:
# =============================================================# H(z) DATA (Cosmic Chronometers)# =============================================================# From Moresco et al. compilationhz_data = np.array([[0.07, 69.0, 19.6],[0.09, 69.0, 12.0],[0.12, 68.6, 26.2],[0.17, 83.0, 8.0],[0.179, 75.0, 4.0],[0.199, 75.0, 5.0],[0.20, 72.9, 29.6],[0.27, 77.0, 14.0],[0.28, 88.8, 36.6],[0.352, 83.0, 14.0],[0.4, 95.0, 17.0],[0.44, 82.6, 7.8],[0.48, 97.0, 62.0],[0.593, 104.0, 13.0],[0.60, 87.9, 6.1],[0.68, 92.0, 8.0],[0.73, 97.3, 7.0],[0.781, 105.0, 12.0],[0.875, 125.0, 17.0],[0.88, 90.0, 40.0],[0.9, 117.0, 23.0],[1.037, 154.0, 20.0],[1.3, 168.0, 17.0],[1.363, 160.0, 33.6],[1.43, 177.0, 18.0],[1.53, 140.0, 14.0],[1.75, 202.0, 40.0],[1.965, 186.5, 50.4],])z_hz = hz_data[:, 0]H_obs = hz_data[:, 1]H_err = hz_data[:, 2]print(f"H(z) data points: {len(z_hz)}")

In [ ]:
# =============================================================# EVAPORATING UNIVERSE MODEL# =============================================================def w_de(z, w0, z_trans):if z_trans <= 0.01:return -1.0if z > z_trans:return -1.0delta_w = w0 - (-1.0)return -1.0 + delta_w * (1 - z/z_trans)**2def H_z(z, H0, Omega_m, w0, z_trans):Omega_de = 1 - Omega_mw = w_de(z, w0, z_trans)rho_de = Omega_de * (1 + z)**(3*(1+w))E_z = np.sqrt(Omega_m * (1+z)**3 + rho_de)return H0 * E_zprint("Model defined.")

In [ ]:
# =============================================================# LIKELIHOOD# =============================================================def log_likelihood(theta):H0, Omega_m, w0, z_trans = thetachi2 = 0for i, z in enumerate(z_hz):H_pred = H_z(z, H0, Omega_m, w0, z_trans)chi2 += ((H_pred - H_obs[i]) / H_err[i])**2return -0.5 * chi2def log_prior(theta):H0, Omega_m, w0, z_trans = thetaif not (60 < H0 < 80): return -np.infif not (0.2 < Omega_m < 0.4): return -np.infif not (-1.5 < w0 < -1.0): return -np.infif not (0.1 < z_trans < 0.5): return -np.infreturn 0.0def log_probability(theta):lp = log_prior(theta)if not np.isfinite(lp):return -np.infreturn lp + log_likelihood(theta)print("Likelihood defined.")

In [ ]:
#=============================================================#RUNMCMC#=============================================================initial=np.array([73.0,0.30,-1.15,0.22])ndim=len(initial)nwalkers=32nsteps=5000pos=initial+1e-3*np.random.randn(nwalkers,ndim)print(f"RunningMCMC:{nwalkers}walkers,{nsteps}steps...")print("Thiswilltake~2-3hoursonColabPro")withPool(n_cores)aspool:sampler=emcee.EnsembleSampler(nwalkers,ndim,log_probability,pool=pool)sampler.run_mcmc(pos,nsteps,progress=True)print("MCMCcomplete!")

In [ ]:
# =============================================================# ANALYZE RESULTS# =============================================================burnin = 1000samples = sampler.get_chain(discard=burnin, flat=True)labels = [r'$H_0$', r'$\Omega_m$', r'$w_0$', r'$z_{trans}$']means = np.mean(samples, axis=0)stds = np.std(samples, axis=0)print("\n" + "="*50)print("H(z) ONLY RESULTS")print("="*50)for i, (label, mean, std) in enumerate(zip(labels, means, stds)):print(f"{label}: {mean:.4f} +/- {std:.4f}")

In [ ]:
# =============================================================# CORNER PLOT# =============================================================fig = corner.corner(samples, labels=labels, quantiles=[0.16, 0.5, 0.84],show_titles=True, title_fmt='.3f')plt.suptitle('H(z) Only Constraints', fontsize=14)plt.tight_layout()plt.savefig('mcmc_hz_only_corner.png', dpi=150)plt.show()

In [ ]:
# =============================================================# SAVE RESULTS# =============================================================results = {"metadata": {"analysis": "MCMC H(z) Only","date": datetime.now().isoformat(),"nwalkers": nwalkers,"nsteps": nsteps,"burnin": burnin},"parameters": {"H0": [float(means[0]), float(stds[0])],"Omega_m": [float(means[1]), float(stds[1])],"w0": [float(means[2]), float(stds[2])],"z_trans": [float(means[3]), float(stds[3])]},"maturity": "Paper Standard","figures": ["mcmc_hz_only_corner.png"]}with open('mcmc_hz_only_results.json', 'w') as f:json.dump(results, f, indent=2)np.save('mcmc_hz_only_chain.npy', samples)print("Saved results!")try:from google.colab import filesfiles.download('mcmc_hz_only_corner.png')files.download('mcmc_hz_only_results.json')files.download('mcmc_hz_only_chain.npy')except:print("Files saved locally.")